In [1]:
import json
from pathlib import Path

In [2]:
ROOT = Path("/lambda/nfs/neel/Research")
SUBSET_NAME = "vqa_v2_balanced_5k"
PRED_DIR = ROOT / "runs" / "dinov2" / SUBSET_NAME
META_PATH = ROOT / "subsets" / SUBSET_NAME / "metadata.jsonl"

In [3]:
def read_jsonl(p: Path):
    rows = []
    with p.open() as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

In [4]:
def normalize(s: str) -> str:
    return " ".join(str(s).strip().lower().split())

In [5]:
def extract_answers(x):
    if x is None:
        return []
    if isinstance(x, list):
        out = []
        for a in x:
            if isinstance(a, str):
                out.append(a)
            elif isinstance(a, dict) and "answer" in a:
                out.append(a["answer"])
            else:
                out.append(str(a))
        return out
    if isinstance(x, dict):
        if "answers" in x:
            return extract_answers(x["answers"])
        if "answer" in x:
            return extract_answers(x["answer"])
    if isinstance(x, str):
        return [x]
    return [str(x)]


In [6]:
def vqa_soft_score(pred: str, gt_answers):
    pred_n = normalize(pred)
    gts = [normalize(a) for a in gt_answers]
    c = sum(1 for a in gts if a == pred_n)
    return min(c / 3.0, 1.0)

In [7]:
def latest_pred_file(pred_dir: Path) -> Path:
    files = sorted(pred_dir.glob("preds_*.jsonl"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not files:
        raise FileNotFoundError(f"No preds_*.jsonl found in {pred_dir}")
    return files[0]

In [8]:
def main():
    if not META_PATH.exists():
        raise FileNotFoundError(META_PATH)

    pred_path = latest_pred_file(PRED_DIR)
    preds = read_jsonl(pred_path)
    rows = read_jsonl(META_PATH)

    gt = {}
    for i, r in enumerate(rows):
        qid = r.get("question_id", r.get("qid", r.get("id", i)))
        ans_key = next((k for k in ["answers", "answer", "gt_answers", "labels"] if k in r), None)
        if ans_key is None:
            raise KeyError(f"No answers key in metadata row keys: {list(r.keys())}")
        gt[qid] = extract_answers(r[ans_key])

    total = 0
    score_sum = 0.0
    for r in preds:
        qid = r.get("question_id")
        pred = r.get("pred_answer", "")
        if qid not in gt:
            continue
        total += 1
        score_sum += vqa_soft_score(pred, gt[qid])

    acc = score_sum / max(total, 1)

    out_dir = PRED_DIR / "eval"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"metrics_{pred_path.stem}.json"
    out_path.write_text(json.dumps({"soft_accuracy": acc, "n_scored": total}, indent=2), encoding="utf-8")

    print(f"Using preds: {pred_path}")
    print(f"VQAv2 soft accuracy: {acc:.4f} (scored {total} questions)")
    print(f"Saved -> {out_path}")

In [9]:
if __name__ == "__main__":
    main()

Using preds: /lambda/nfs/neel/Research/runs/dinov2/vqa_v2_balanced_5k/preds_dinov2_vitb14.jsonl
VQAv2 soft accuracy: 0.1266 (scored 5000 questions)
Saved -> /lambda/nfs/neel/Research/runs/dinov2/vqa_v2_balanced_5k/eval/metrics_preds_dinov2_vitb14.json
